<a href="https://colab.research.google.com/github/Stdcoders/Graph-RAG/blob/main/GraphRAG_L6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/genaiconference/Agentic_KAG_Workshop_DHS_2026.git

Cloning into 'Agentic_KAG_Workshop_DHS_2026'...
remote: Enumerating objects: 403, done.
remote: Counting objects: 100% (179/179), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 403 (delta 125), reused 57 (delta 55), pack-reused 224 (from 2)
Receiving objects: 100% (403/403), 13.34 MiB | 16.51 MiB/s, done.
Resolving deltas: 100% (217/217), done.


In [ ]:
%pip install --quiet -r /content/Agentic_KAG_Workshop_DHS_2026/requirements.txt

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.7/263.7 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.2/58.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.0/358.0 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.8/85.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 669.4/669.4 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.9 MB/s eta 0:00:00
   ━━━━━━

In [ ]:
import os
import logging

# Silence the specific Neo4j notification logger
logging.getLogger("neo4j.notifications").setLevel(logging.ERROR)


os.chdir('/content/Agentic_KAG_Workshop_DHS_2026/')

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    print("error reading env details")
    pass

# --- Neo4j Sandbox ---
NEO4J_URI      = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE')

# --- OpenAI ---
os.environ.setdefault(
    'NVIDIA_API_KEY',
    os.getenv('NVIDIA_API_KEY')
)

print('NEO4J_URI :', NEO4J_URI)
print('NVIDIA key set:', bool(os.environ.get('NVIDIA_API_KEY')))

NEO4J_URI : neo4j+s://313964e6.databases.neo4j.io
NVIDIA key set: True


In [ ]:
from neo4j_graphrag.llm import OpenAILLM
NVIDIA_BASE_URL = "https://integrate.api.nvidia.com/v1"
llm = OpenAILLM(
    model_name="nvidia/nemotron-3-super-120b-a12b",
    model_params={
        "response_format": {"type": "json_object"},
    },
    base_url=NVIDIA_BASE_URL,
    api_key=os.getenv("NVIDIA_API_KEY"),
)

In [ ]:
from neo4j import GraphDatabase

class MovieCommunitySummarizer:
    def __init__(self, driver,llm=None,graph_name='movie_graph'):
        self.driver=driver
        self.llm=llm
        self.graph_name=graph_name

    def project_graph(self):
        # Discover which labels and relationship types ACTUALLY exist in the DB.
        # GDS fails the whole projection if we list a relationship type that isn't
        # present, so we build the projection dynamically from the live schema.
        with self.driver.session() as s:
            labels = [r["label"] for r in s.run("CALL db.labels() YIELD label RETURN label")]
            rel_types = [
                r["relationshipType"]
                for r in s.run(
                    "CALL db.relationshipTypes() YIELD relationshipType RETURN relationshipType"
                )
            ]

        # Exclude the :Community label we write later so it doesn't pollute the projection
        labels = [l for l in labels if l != "Community"]

        if not labels or not rel_types:
            raise ValueError(
                "No labels/relationships found to project. "
                "Did you run the sample-data loader (Step 1)?"
            )

        rel_projection = {rt: {"orientation": "UNDIRECTED"} for rt in rel_types}

        with self.driver.session() as s:
            try:
                # yield an explicit column so we don't trigger the deprecated `schema` return
                s.run(
                    "CALL gds.graph.drop($g, false) YIELD graphName RETURN graphName",
                    g=self.graph_name,
                ).consume()
            except Exception:
                pass
            # .consume() forces the projection to actually run before we continue
            s.run(
                "CALL gds.graph.project(
labels, $rels)",
                graph=self.graph_name,
                labels=labels,
                rels=rel_projection,
            ).consume()

        print(f"   Projected labels: {labels}")
        print(f"   Projected relationships: {rel_types}")

    def run_leiden(self):
        # IMPORTANT: consume the result so the write-back actually executes & commits.
        # Without .consume()/.data(), the auto-commit transaction may be discarded
        # before Leiden writes the `community` property -> 0 communities downstream.
        query = """
        CALL gds.leiden.write($g, {writeProperty: 'community'})
        YIELD communityCount, nodePropertiesWritten, modularity
        RETURN communityCount, nodePropertiesWritten, modularity
        """
        with self.driver.session() as s:
            record = s.run(query, g=self.graph_name).single()
        print(
            f"   Leiden wrote 'community' to {record['nodePropertiesWritten']} nodes "
            f"across {record['communityCount']} communities "
            f"(modularity={record['modularity']:.4f})"
        )
        return record

    def get_communities(self, member_limit=60):
        # Return COMPACT community data so we don't blow the LLM token limit.
        # - Only node name/title + primary label (not full properties)
        # - Relationship info as type counts (not every relationship w/ full targets)
        # `member_limit` caps how many member names we keep per community.
        q = '''
MATCH (n)
WHERE n.community IS NOT NULL
WITH n.community AS community,
     collect(DISTINCT coalesce(n.title, n.name, head(labels(n))))[0..$limit] AS members,
     count(DISTINCT n) AS size
RETURN community, members, size
ORDER BY size DESC
'''
        rels_q = '''
MATCH (n)-[r]-(m)
WHERE n.community =
cid
RETURN type(r) AS rel_type, count(DISTINCT r) AS cnt
ORDER BY cnt DESC
'''
        with self.driver.session() as s:
            rows = [x.data() for x in s.run(q, limit=member_limit)]
            for row in rows:
                row['rel_counts'] = {
                    r['rel_type']: r['cnt']
                    for r in s.run(rels_q, cid=row['community'])
                }
        return rows

    async def summarize_community(self, c):
        # Build a compact prompt from the trimmed community data.
        members = c["members"]
        shown = ", ".join(str(m) for m in members)
        more = c["size"] - len(members)
        if more > 0:
            shown += f", ... (+{more} more)"
        rels = ", ".join(f"{k}:{v}" for k, v in c.get("rel_counts", {}).items()) or "none"
        prompt = (
            "You are summarizing a community from a Movie Knowledge Graph.\n"
            f"This community has {c['size']} members.\n"
            f"Relationship types (with counts): {rels}\n"
            f"Sample members: {shown}\n\n"
            "In 2-3 sentences, describe the common theme of this community "
            "(e.g. genre, language/industry, franchise, key people)."
        )
        if self.llm is None:
            return prompt
        result = await self.llm.ainvoke(prompt)
        return result.content if hasattr(result, 'content') else str(result)

    async def summarize_all(self):
        out = []
        for c in self.get_communities():
            out.append((c['community'], await self.summarize_community(c)))
        return out

    def save_summaries(self, summaries):
        # 1) Upsert the Community node with a STABLE, namespaced id plus useful
        #    properties (title, size) — not just the summary.
        #    - id: namespaced (e.g. "movie-community-3") so ids never collide
        #          with other pipelines and MERGE stays idempotent.
        #    - community_no: the raw Leiden integer, used to link members.
        #    - title: a short label derived from the first line/sentence of the summary.
        #    - size: number of member entities in the community.
        # 2) LINK every entity with this community id to the Community node, so the
        #    graph shows edges (entity)-[:IN_COMMUNITY]->(Community).
        upsert_q = '''
MERGE (c:Community {id:node_id}) SET c.community_no =
cid,
    c.summary =
title
'''
        link_q = '''
MATCH (c:Community {id:node_id}) MATCH (n) WHERE n.community =
cid AND NOT n:Community
WITH c, collect(n) AS members
SET c.size = size(members)
FOREACH (n IN members | MERGE (n)-[:IN_COMMUNITY]->(c))
'''
        with self.driver.session() as s:
            for cid, summary in summaries:
                node_id = f"movie-community-{cid}"
                # Derive a short title from the first sentence/line of the summary.
                title = summary.strip().split("\n")[0].split(". ")[0][:80]
                s.run(upsert_q, node_id=node_id, cid=cid,
                      summary=summary, title=title).consume()
                s.run(link_q, node_id=node_id, cid=cid).consume()
        print(f"   Linked members to {len(summaries)} Community nodes via :IN_COMMUNITY")

    def drop_projection(self):
        with self.driver.session() as s:
            s.run(
                "CALL gds.graph.drop($g, false) YIELD graphName RETURN graphName",
                g=self.graph_name,
            ).consume()